In [1]:
from graphlib import TopologicalSorter
import copy

GOAL = {"text": "Plan a one-day trip and be back by 21:00 within Rs 600",
        "budget": 600, "depart": 8, "back_by": 21, "min_stay_h": 4}
FOOD = 300   # fixed cost of food for the day (assumed)

# Static knowledge (the agent knows this)
PLACES = {
    "Hill Fort":   {"km": 60, "fee": 50,  "outdoor": True,  "interest": 8},
    "Lake Park":   {"km": 25, "fee": 0,   "outdoor": True,  "interest": 6},
    "Museum Town": {"km": 90, "fee": 100, "outdoor": False, "interest": 7},
}
MODES = {"train": {"rs_per_km": 0.5, "kmph": 50},
         "bus":   {"rs_per_km": 1.2, "kmph": 35},
         "cab":   {"rs_per_km": 4.0, "kmph": 50}}

# Live world (the agent can only see this through tools, and it can change)
ENV = {}
def reset_world():
    ENV.clear()
    ENV["weather"] = {p: "clear" for p in PLACES}
    ENV["cancelled"] = set()

# ---- Tools ----
def tool_weather(place):
    return ENV["weather"][place]

def tool_transport_status(mode):
    return "running" if mode not in ENV["cancelled"] else "cancelled"

def tool_book(mode, place):
    if mode in ENV["cancelled"]:
        return f"FAILED: {mode} is cancelled today"
    return f"BOOKED: {mode} to {place}"

In [2]:
SUBTASKS = {   # sub-task: set of sub-tasks that must finish first
    "choose_option":       set(),
    "check_weather":       {"choose_option"},
    "check_transport":     {"choose_option"},
    "book_transport":      {"check_weather", "check_transport"},
    "final_weather_check": {"book_transport"},
    "write_itinerary":     {"final_weather_check"},
}
ORDER = list(TopologicalSorter(SUBTASKS).static_order())

WHY = {
    "choose_option":       "I cannot check weather or tickets until I know which place and which transport I am considering.",
    "check_weather":       "Outdoor places are risky in rain, so the weather must be known before I pay for anything.",
    "check_transport":     "The chosen vehicle must actually be running today.",
    "book_transport":      "Booking costs money, so it comes only after weather and transport are confirmed.",
    "final_weather_check": "Weather can change after booking, so I check once more before finalising.",
    "write_itinerary":     "The itinerary is only written when every earlier check has passed.",
}
print("GOAL:", GOAL["text"], "\n")
for n, task in enumerate(ORDER, 1):
    deps = ", ".join(sorted(SUBTASKS[task])) or "none"
    print(f"Thought {n}: do '{task}' (needs: {deps}). {WHY[task]}")

GOAL: Plan a one-day trip and be back by 21:00 within Rs 600 

Thought 1: do 'choose_option' (needs: none). I cannot check weather or tickets until I know which place and which transport I am considering.
Thought 2: do 'check_weather' (needs: choose_option). Outdoor places are risky in rain, so the weather must be known before I pay for anything.
Thought 3: do 'check_transport' (needs: choose_option). The chosen vehicle must actually be running today.
Thought 4: do 'book_transport' (needs: check_transport, check_weather). Booking costs money, so it comes only after weather and transport are confirmed.
Thought 5: do 'final_weather_check' (needs: book_transport). Weather can change after booking, so I check once more before finalising.
Thought 6: do 'write_itinerary' (needs: final_weather_check). The itinerary is only written when every earlier check has passed.


In [3]:
def evaluate(place, mode, facts):
    p, m = PLACES[place], MODES[mode]
    cost = round(2 * p["km"] * m["rs_per_km"]) + p["fee"] + FOOD
    hours = round(2 * p["km"] / m["kmph"], 1)
    problems = []
    if mode in facts["cancelled"]:
        problems.append("transport cancelled")
    if p["outdoor"] and facts["weather"].get(place) == "rain":
        problems.append("rain at outdoor place")
    if hours + GOAL["min_stay_h"] > GOAL["back_by"] - GOAL["depart"]:
        problems.append("not enough time")
    over_budget = cost - GOAL["budget"]
    score = round(10 * p["interest"] - 20 * cost / GOAL["budget"] - 3 * hours, 1)
    return {"place": place, "mode": mode, "cost": cost, "hours": hours, "score": score,
            "problems": problems, "over_budget": max(over_budget, 0)}

def tot_search(facts, verbose=True):
    branches = [evaluate(pl, md, facts) for pl in PLACES for md in MODES]
    for b in branches:
        # budget is a soft problem so we can still report the "closest" option
        b["valid"] = not b["problems"] and b["over_budget"] == 0
    if verbose:
        for pl in PLACES:
            print(f"   {pl}")
            for b in [x for x in branches if x["place"] == pl]:
                if b["valid"]:
                    print(f"      - {b['mode']:5} cost Rs {b['cost']:>4}, {b['hours']}h  -> score {b['score']}")
                else:
                    why = b["problems"] + ([f"over budget by Rs {b['over_budget']}"] if b["over_budget"] else [])
                    print(f"      x {b['mode']:5} cost Rs {b['cost']:>4}, {b['hours']}h  -> pruned ({', '.join(why)})")
    valid = [b for b in branches if b["valid"]]
    if valid:
        return max(valid, key=lambda b: b["score"]), None
    # nothing valid: is there a branch that fails ONLY because of budget?
    near = [b for b in branches if not b["problems"] and b["over_budget"] > 0]
    return None, (min(near, key=lambda b: b["over_budget"]) if near else None)

print("Tree of thoughts with the starting beliefs (everything looks fine):\n")
best, _ = tot_search({"cancelled": set(), "weather": {}})
print("\nChosen branch:", best["place"], "by", best["mode"], "| score", best["score"])

Tree of thoughts with the starting beliefs (everything looks fine):

   Hill Fort
      - train cost Rs  410, 2.4h  -> score 59.1
      - bus   cost Rs  494, 3.4h  -> score 53.3
      x cab   cost Rs  830, 2.4h  -> pruned (over budget by Rs 230)
   Lake Park
      - train cost Rs  325, 1.0h  -> score 46.2
      - bus   cost Rs  360, 1.4h  -> score 43.8
      - cab   cost Rs  500, 1.0h  -> score 40.3
   Museum Town
      - train cost Rs  490, 3.6h  -> score 42.9
      x bus   cost Rs  616, 5.1h  -> pruned (over budget by Rs 16)
      x cab   cost Rs 1120, 3.6h  -> pruned (over budget by Rs 520)

Chosen branch: Hill Fort by train | score 59.1


In [4]:
def react_agent(events, replan=True, max_replans=3):
    """events: {sub-task name: change applied to the world just before that sub-task runs (fires once)}"""
    reset_world()
    events = copy.deepcopy(events)
    facts = {"cancelled": set(), "weather": {}}
    plan, replans, i, step = None, [], 0, 0

    def violated(reason):
        """Handle a broken assumption. Returns True if the run can continue (replan)."""
        nonlocal i
        print(f"      !! ASSUMPTION VIOLATED: {reason}")
        if not replan:
            print("      !! replanning is switched off -> the agent cannot recover")
            return False
        if len(replans) >= max_replans:
            print("      !! too many replans")
            return False
        replans.append({"reason": reason, "old_plan": f"{plan['place']} by {plan['mode']}"})
        print("      Thought: my plan rested on a belief that is now false. I update my beliefs and search the tree again.")
        i = 0                                  # go back to choose_option with the updated beliefs
        return True

    while i < len(ORDER):
        task = ORDER[i]
        # the real world may change right before this step
        change = events.pop(task, None)
        if change:
            ENV["cancelled"] |= set(change.get("cancel", []))
            ENV["weather"].update(change.get("weather", {}))
        step += 1
        print(f"Step {step}: {task}")

        if task == "choose_option":
            print("   Thought: search the tree with what I currently believe.")
            plan, near = tot_search(facts, verbose=False)
            print("   Action : tot_search(beliefs)")
            if plan is None:
                if near:
                    msg = (f"NEEDS HUMAN APPROVAL: the closest option is {near['place']} by {near['mode']} "
                           f"(Rs {near['cost']}), which is Rs {near['over_budget']} over the Rs {GOAL['budget']} budget")
                else:
                    msg = "NO VALID PLAN: every option breaks a hard constraint"
                print("   Observation:", msg)
                return {"status": "STOPPED", "detail": msg, "replans": replans, "plan": None}
            print(f"   Observation: best branch = {plan['place']} by {plan['mode']} (cost Rs {plan['cost']}, {plan['hours']}h, score {plan['score']})")

        elif task in ("check_weather", "final_weather_check"):
            print(f"   Thought: is it safe to go to {plan['place']}? (assumption: no rain if the place is outdoor)")
            w = tool_weather(plan["place"])
            print(f"   Action : tool_weather('{plan['place']}')")
            print(f"   Observation: {w}")
            facts["weather"][plan["place"]] = w
            if PLACES[plan["place"]]["outdoor"] and w == "rain":
                if not violated(f"rain at outdoor place {plan['place']}"):
                    return {"status": "FAILED", "detail": "rain at an outdoor place", "replans": replans, "plan": plan}
                continue

        elif task == "check_transport":
            print(f"   Thought: is the {plan['mode']} running today? (assumption: yes)")
            s = tool_transport_status(plan["mode"])
            print(f"   Action : tool_transport_status('{plan['mode']}')")
            print(f"   Observation: {s}")
            if s == "cancelled":
                facts["cancelled"].add(plan["mode"])
                if not violated(f"{plan['mode']} is cancelled"):
                    return {"status": "FAILED", "detail": f"{plan['mode']} cancelled", "replans": replans, "plan": plan}
                continue

        elif task == "book_transport":
            print(f"   Thought: both checks passed, so I can book the {plan['mode']}.")
            r = tool_book(plan["mode"], plan["place"])
            print(f"   Action : tool_book('{plan['mode']}', '{plan['place']}')")
            print(f"   Observation: {r}")
            if r.startswith("FAILED"):
                facts["cancelled"].add(plan["mode"])
                if not violated(f"booking failed, {plan['mode']} is cancelled"):
                    return {"status": "FAILED", "detail": r, "replans": replans, "plan": plan}
                continue

        elif task == "write_itinerary":
            print("   Thought: all checks passed, so I can write the itinerary.")
            print("   Action : write_itinerary(plan)")
            print(f"   Observation: 08:00 depart by {plan['mode']} -> {plan['place']} (min {GOAL['min_stay_h']}h stay) -> be home before 21:00. "
                  f"Estimated cost Rs {plan['cost']} (estimate, not a receipt).")
        print()
        i += 1
    return {"status": "DONE", "detail": "itinerary written", "replans": replans, "plan": plan}

In [5]:
resA = react_agent(events={})
print("RESULT:", resA["status"], "| replans:", len(resA["replans"]))

Step 1: choose_option
   Thought: search the tree with what I currently believe.
   Action : tot_search(beliefs)
   Observation: best branch = Hill Fort by train (cost Rs 410, 2.4h, score 59.1)

Step 2: check_weather
   Thought: is it safe to go to Hill Fort? (assumption: no rain if the place is outdoor)
   Action : tool_weather('Hill Fort')
   Observation: clear

Step 3: check_transport
   Thought: is the train running today? (assumption: yes)
   Action : tool_transport_status('train')
   Observation: running

Step 4: book_transport
   Thought: both checks passed, so I can book the train.
   Action : tool_book('train', 'Hill Fort')
   Observation: BOOKED: train to Hill Fort

Step 5: final_weather_check
   Thought: is it safe to go to Hill Fort? (assumption: no rain if the place is outdoor)
   Action : tool_weather('Hill Fort')
   Observation: clear

Step 6: write_itinerary
   Thought: all checks passed, so I can write the itinerary.
   Action : write_itinerary(plan)
   Observation: 08

In [6]:
EVENTS_B = {"book_transport": {"cancel": ["train"]}}
print("##### B1: replanning OFF (plan once, never adapt) #####\n")
resB_off = react_agent(EVENTS_B, replan=False)
print("RESULT:", resB_off["status"], "-", resB_off["detail"])
print("\n##### B2: replanning ON #####\n")
resB_on = react_agent(EVENTS_B, replan=True)
print("RESULT:", resB_on["status"], "| replans:", len(resB_on["replans"]))

##### B1: replanning OFF (plan once, never adapt) #####

Step 1: choose_option
   Thought: search the tree with what I currently believe.
   Action : tot_search(beliefs)
   Observation: best branch = Hill Fort by train (cost Rs 410, 2.4h, score 59.1)

Step 2: check_weather
   Thought: is it safe to go to Hill Fort? (assumption: no rain if the place is outdoor)
   Action : tool_weather('Hill Fort')
   Observation: clear

Step 3: check_transport
   Thought: is the train running today? (assumption: yes)
   Action : tool_transport_status('train')
   Observation: running

Step 4: book_transport
   Thought: both checks passed, so I can book the train.
   Action : tool_book('train', 'Hill Fort')
   Observation: FAILED: train is cancelled today
      !! ASSUMPTION VIOLATED: booking failed, train is cancelled
      !! replanning is switched off -> the agent cannot recover
RESULT: FAILED - FAILED: train is cancelled today

##### B2: replanning ON #####

Step 1: choose_option
   Thought: search t

In [7]:
EVENTS_C = {"book_transport": {"cancel": ["train"]},
            "final_weather_check": {"weather": {"Hill Fort": "rain"}}}
resC = react_agent(EVENTS_C, replan=True)
print("RESULT:", resC["status"], "|", resC["detail"])

Step 1: choose_option
   Thought: search the tree with what I currently believe.
   Action : tot_search(beliefs)
   Observation: best branch = Hill Fort by train (cost Rs 410, 2.4h, score 59.1)

Step 2: check_weather
   Thought: is it safe to go to Hill Fort? (assumption: no rain if the place is outdoor)
   Action : tool_weather('Hill Fort')
   Observation: clear

Step 3: check_transport
   Thought: is the train running today? (assumption: yes)
   Action : tool_transport_status('train')
   Observation: running

Step 4: book_transport
   Thought: both checks passed, so I can book the train.
   Action : tool_book('train', 'Hill Fort')
   Observation: FAILED: train is cancelled today
      !! ASSUMPTION VIOLATED: booking failed, train is cancelled
      Thought: my plan rested on a belief that is now false. I update my beliefs and search the tree again.
Step 5: choose_option
   Thought: search the tree with what I currently believe.
   Action : tot_search(beliefs)
   Observation: best bra

In [8]:
EVENTS_D = {"book_transport": {"cancel": ["train"]},
            "final_weather_check": {"weather": {"Hill Fort": "rain", "Lake Park": "rain"}}}
resD = react_agent(EVENTS_D, replan=True)
print("RESULT:", resD["status"], "|", resD["detail"])

Step 1: choose_option
   Thought: search the tree with what I currently believe.
   Action : tot_search(beliefs)
   Observation: best branch = Hill Fort by train (cost Rs 410, 2.4h, score 59.1)

Step 2: check_weather
   Thought: is it safe to go to Hill Fort? (assumption: no rain if the place is outdoor)
   Action : tool_weather('Hill Fort')
   Observation: clear

Step 3: check_transport
   Thought: is the train running today? (assumption: yes)
   Action : tool_transport_status('train')
   Observation: running

Step 4: book_transport
   Thought: both checks passed, so I can book the train.
   Action : tool_book('train', 'Hill Fort')
   Observation: FAILED: train is cancelled today
      !! ASSUMPTION VIOLATED: booking failed, train is cancelled
      Thought: my plan rested on a belief that is now false. I update my beliefs and search the tree again.
Step 5: choose_option
   Thought: search the tree with what I currently believe.
   Action : tot_search(beliefs)
   Observation: best bra

In [9]:
for name, res in (("B2", resB_on), ("C", resC), ("D", resD)):
    print(f"Scenario {name}: final status = {res['status']}")
    for n, r in enumerate(res["replans"], 1):
        print(f"  Replan {n}: because '{r['reason']}' (plan before: {r['old_plan']})")
    if res["plan"]:
        print(f"  Last plan: {res['plan']['place']} by {res['plan']['mode']}, Rs {res['plan']['cost']}")
    print()

Scenario B2: final status = DONE
  Replan 1: because 'booking failed, train is cancelled' (plan before: Hill Fort by train)
  Last plan: Hill Fort by bus, Rs 494

Scenario C: final status = DONE
  Replan 1: because 'booking failed, train is cancelled' (plan before: Hill Fort by train)
  Replan 2: because 'rain at outdoor place Hill Fort' (plan before: Hill Fort by bus)
  Last plan: Lake Park by bus, Rs 360

Scenario D: final status = STOPPED
  Replan 1: because 'booking failed, train is cancelled' (plan before: Hill Fort by train)
  Replan 2: because 'rain at outdoor place Hill Fort' (plan before: Hill Fort by bus)
  Replan 3: because 'rain at outdoor place Lake Park' (plan before: Lake Park by bus)



In [10]:
rows = [("A  no change",            resA),
        ("B1 train cancelled, no replan", resB_off),
        ("B2 train cancelled, replan",    resB_on),
        ("C  train + rain at Hill Fort",  resC),
        ("D  train + rain everywhere",    resD)]
print(f'{"scenario":32} {"outcome":9} {"replans":8} final plan')
for name, r in rows:
    p = r["plan"]
    final = f"{p['place']} by {p['mode']} (Rs {p['cost']})" if p and r["status"] == "DONE" else r["detail"][:60]
    print(f'{name:32} {r["status"]:9} {len(r["replans"]):<8} {final}')

scenario                         outcome   replans  final plan
A  no change                     DONE      0        Hill Fort by train (Rs 410)
B1 train cancelled, no replan    FAILED    0        FAILED: train is cancelled today
B2 train cancelled, replan       DONE      1        Hill Fort by bus (Rs 494)
C  train + rain at Hill Fort     DONE      2        Lake Park by bus (Rs 360)
D  train + rain everywhere       STOPPED   3        NEEDS HUMAN APPROVAL: the closest option is Museum Town by b
